# 07c — Model V2 Training Experiments

Full-data V2 experimentation around the proven `07a` baseline recipe.

**Baseline anchor:**
- Dataset: `2026-06-03_008`
- Split: `price_stratified`
- Target: `price_egp_log`
- Features: V1 only (15 features)

**Experiments:**
- WeightedEnsemble baseline parity
- WeightedEnsemble tuned variant
- XGBoost tuned + sample-weighted challenger
- 5-quantile regression for interval coverage

> **Execution note:** This notebook runs on the **full dataset** with a reduced shortlist of high-value experiments. Optuna tuning still takes time. Each experiment is cached to `_partial_runs__/`. Delete that folder after logic changes to avoid stale caches.

## 1) Imports & Configuration

Load shared helpers and lock the baseline configuration explicitly.

In [4]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'v2_experiment_utils.py').exists():
    NOTEBOOK_DIR = Path('/home/mo-seif/Documents/GP/ml-service/notebooks')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from v2_experiment_utils import (
    RANDOM_STATE,
    DEFAULT_QUANTILES_5,
    XGB_BASE_PARAMS,
    LGBM_BASE_PARAMS,
    load_processed_dataset,
    build_primary_splits,
    split_train_val_indices,
    build_feature_spec,
    make_lgbm_frame,
    fit_label_encoders,
    transform_with_label_encoders,
    tune_lgbm_quantile,
    tune_xgb_quantile,
    train_lgbm_quantile_models,
    train_xgb_quantile_models,
    predict_lgbm_quantiles,
    predict_xgb_quantiles,
    blend_prediction_maps,
    evaluate_point_and_interval,
    per_make_model_mape_df,
    sample_weights_from_price,
    run_kfold_per_make_model_cv,
    make_experiment_dir,
)

pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
np.set_printoptions(suppress=True)

In [5]:
DATASET_TAG = '2026-06-03_008'
SPLIT_NAME = 'price_stratified'
TARGET_COL = 'price_egp_log'
FEATURE_LIST = []

OPTUNA_TRIALS = 50
MAX_BOOST_ROUNDS = 1200
EARLY_STOPPING_ROUNDS = 40

RUN_MODE_TAG = (
    f'{DATASET_TAG}__{SPLIT_NAME}__{TARGET_COL}'
    f'__full_opt{OPTUNA_TRIALS}_r{MAX_BOOST_ROUNDS}'
)
OUT_DIR = make_experiment_dir('07c_model_v2_training_experiments')
PARTIAL_DIR = OUT_DIR / f'_partial_runs__{RUN_MODE_TAG}'
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset tag:', DATASET_TAG)
print('Split:', SPLIT_NAME)
print('Target:', TARGET_COL)
print('Features: V1 only (no Plan 2)')
print('Optuna trials:', OPTUNA_TRIALS)
print('Max boost rounds:', MAX_BOOST_ROUNDS)
print('Output dir:', OUT_DIR)

Dataset tag: 2026-06-03_008
Split: price_stratified
Target: price_egp_log
Features: V1 only (no Plan 2)
Optuna trials: 50
Max boost rounds: 1200
Output dir: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments


## 2) 07a Baseline Reference

Load the trusted 07a baseline results for comparison.

In [6]:
BASELINE_DIR = Path('/home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07a_dataset_baselines')
baseline_summary_path = BASELINE_DIR / 'baseline_results_summary.csv'

if baseline_summary_path.exists():
    baseline_df = pd.read_csv(baseline_summary_path)
    baseline_row = baseline_df[
        (baseline_df.get('dataset_tag', '') == DATASET_TAG) &
        (baseline_df.get('split_name', '') == SPLIT_NAME) &
        (baseline_df.get('target_col', '') == TARGET_COL)
    ]
    if len(baseline_row) == 0:
        baseline_row = baseline_df
    display(baseline_row)
else:
    print('07a baseline summary not found at:', baseline_summary_path)
    print('Expected 07a baseline: WeightedEnsemble, MAPE ~11.33%')

baseline_mape = 11.33  # from 07a REPORT.md
print(f'Baseline MAPE reference: {baseline_mape}%')

,dataset_tag,split_name,target_col,framework,MAE,RMSE,R2,MAPE_pct,Within_10pct,Within_15pct,Coverage_80_pct,Coverage_90_pct,Mean_width_pct
0,2026-06-03_008,price_stratified,price_egp_log,WeightedEnsemble,"150,593.8633","605,558.6220",0.8875,11.3258,61.2400,76.7600,85.7600,85.7600,61.5454


Baseline MAPE reference: 11.33%


## 3) Load Dataset & Reconstruct Baseline Split

Load the processed dataset and reconstruct the `price_stratified` split used in 07a.

In [7]:
df_full = load_processed_dataset(DATASET_TAG)
split_map = build_primary_splits(df_full)
train_idx, test_idx = split_map[SPLIT_NAME]
train_idx_fit, val_idx = split_train_val_indices(train_idx, val_fraction=0.10, random_state=RANDOM_STATE)

train_df = df_full.loc[train_idx_fit].copy()
val_df = df_full.loc[val_idx].copy()
test_df = df_full.loc[test_idx].copy()

print('Dataset tag:', DATASET_TAG)
print('Target:', TARGET_COL)
print('Split:', SPLIT_NAME)
print(f'Train rows: {len(train_df):,}')
print(f'Val rows: {len(val_df):,}')
print(f'Test rows: {len(test_df):,}')

Dataset tag: 2026-06-03_008
Target: price_egp_log
Split: price_stratified
Train rows: 8,999
Val rows: 999
Test rows: 2,500


In [8]:
NUM_COLS, CAT_COLS, FEATURE_COLS = build_feature_spec(FEATURE_LIST)

print('Feature list:', FEATURE_LIST)
print('Feature count:', len(FEATURE_COLS))

Feature list: []
Feature count: 15


## 4) Prepare Training Matrices

Build LightGBM (native categorical) and XGBoost (label-encoded) input frames.

In [9]:
X_train_lgb = make_lgbm_frame(train_df, FEATURE_COLS, CAT_COLS)
X_val_lgb = make_lgbm_frame(val_df, FEATURE_COLS, CAT_COLS)
X_test_lgb = make_lgbm_frame(test_df, FEATURE_COLS, CAT_COLS)

label_encoders = fit_label_encoders(train_df, CAT_COLS)
X_train_xgb = transform_with_label_encoders(train_df, FEATURE_COLS, CAT_COLS, label_encoders)
X_val_xgb = transform_with_label_encoders(val_df, FEATURE_COLS, CAT_COLS, label_encoders)
X_test_xgb = transform_with_label_encoders(test_df, FEATURE_COLS, CAT_COLS, label_encoders)

print('LightGBM frames: train', X_train_lgb.shape, 'val', X_val_lgb.shape, 'test', X_test_lgb.shape)
print('XGBoost frames:  train', X_train_xgb.shape, 'val', X_val_xgb.shape, 'test', X_test_xgb.shape)

LightGBM frames: train (8999, 15) val (999, 15) test (2500, 15)
XGBoost frames:  train (8999, 15) val (999, 15) test (2500, 15)


In [10]:
train_weights = sample_weights_from_price(train_df['price_egp'])
val_weights = sample_weights_from_price(val_df['price_egp'])
is_log_target = TARGET_COL == 'price_egp_log'

print('Training sample weights ready.')
print('Validation sample weights ready.')
print('Is log target:', is_log_target)

Training sample weights ready.
Validation sample weights ready.
Is log target: True


## 5) Optuna Tuning Configuration

Tune the median quantile for both frameworks. Full budget; expect several minutes per framework.

In [8]:
xgb_tuned_params, xgb_study = tune_xgb_quantile(
    X_train_xgb,
    train_df[TARGET_COL],
    X_val_xgb,
    val_df[TARGET_COL],
    alpha=0.50,
    n_trials=OPTUNA_TRIALS,
    n_estimators_max=MAX_BOOST_ROUNDS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    random_state=RANDOM_STATE,
)

print('Best tuned XGBoost params:')
for k, v in sorted(xgb_tuned_params.items()):
    print(f'  {k}: {v}')

Best tuned XGBoost params:
  colsample_bytree: 0.6846439560677529
  gamma: 0.26366384594319553
  learning_rate: 0.029861581602051584
  max_bin: 512
  max_depth: 12
  min_child_weight: 10
  reg_alpha: 4.580877549064464e-05
  reg_lambda: 0.0005053856650861515
  subsample: 0.6841787747408874


In [10]:
lgbm_tuned_params, lgbm_study = tune_lgbm_quantile(
    X_train_lgb,
    train_df[TARGET_COL],
    X_val_lgb,
    val_df[TARGET_COL],
    CAT_COLS,
    alpha=0.50,
    n_trials=OPTUNA_TRIALS,
    n_estimators_max=MAX_BOOST_ROUNDS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    random_state=RANDOM_STATE,
)

print('Best tuned LightGBM params:')
for k, v in sorted(lgbm_tuned_params.items()):
    print(f'  {k}: {v}')

Best tuned LightGBM params:
  colsample_bytree: 0.8038323196663962
  learning_rate: 0.1023489273238644
  max_depth: 9
  min_child_samples: 17
  min_split_gain: 0.08654546077907388
  num_leaves: 22
  reg_alpha: 0.004676612441451472
  reg_lambda: 6.994262975013571e-05
  subsample: 0.9311080741209209


## 6) Define V2 Experiment Ladder

Use a reduced shortlist centered on the strongest baseline-style candidate and one serious single-model challenger:

| # | Experiment | Framework | Params | Weights | Quantiles |
|---|------------|-----------|--------|---------|-----------|
| 1 | `ensemble_baseline_5q` | WeightedEnsemble | Default | No | 5 |
| 2 | `ensemble_tuned_5q` | WeightedEnsemble | Tuned | No | 5 |
| 3 | `xgb_tuned_weighted_5q` | XGBoost | Tuned | Yes | 5 |

**Question:** Does the tuned ensemble or the tuned weighted XGBoost challenger beat the trusted 07a baseline (~11.33% MAPE)?

In [8]:
EXPERIMENTS = [
    {'experiment_name': 'ensemble_baseline_5q', 'framework': 'WeightedEnsemble', 'quantiles': DEFAULT_QUANTILES_5, 'use_weights': False, 'use_tuned': False},
    {'experiment_name': 'ensemble_tuned_5q', 'framework': 'WeightedEnsemble', 'quantiles': DEFAULT_QUANTILES_5, 'use_weights': False, 'use_tuned': True},
    {'experiment_name': 'xgb_tuned_weighted_5q', 'framework': 'XGBoost', 'quantiles': DEFAULT_QUANTILES_5, 'use_weights': True, 'use_tuned': True},
]
print(f'Defined {len(EXPERIMENTS)} experiments')
for exp in EXPERIMENTS:
    print(f"  {exp['experiment_name']}: {exp['framework']} tuned={exp['use_tuned']} weights={exp['use_weights']}")

Defined 3 experiments
  ensemble_baseline_5q: WeightedEnsemble tuned=False weights=False
  ensemble_tuned_5q: WeightedEnsemble tuned=True weights=False
  xgb_tuned_weighted_5q: XGBoost tuned=True weights=True


## 7) Experiment Runner

Helper function to train and evaluate one experiment. Handles caching per experiment.

In [11]:
def run_single_experiment(experiment: dict) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    """Train one experiment and return (metrics_df, pred_map, diag_df)."""
    name = experiment['experiment_name']
    framework = experiment['framework']
    quantiles = experiment['quantiles']
    use_weights = experiment['use_weights']
    use_tuned = experiment['use_tuned']
    
    train_w = train_weights if use_weights else None
    val_w = val_weights if use_weights else None
    
    xgb_params = ({**XGB_BASE_PARAMS, **xgb_tuned_params, 'seed': RANDOM_STATE} if use_tuned else {**XGB_BASE_PARAMS, 'seed': RANDOM_STATE})
    lgbm_params = ({**LGBM_BASE_PARAMS, **lgbm_tuned_params, 'seed': RANDOM_STATE} if use_tuned else {**LGBM_BASE_PARAMS, 'seed': RANDOM_STATE})
    
    pred_map: dict[str, np.ndarray] = {}
    
    if framework in ('XGBoost', 'WeightedEnsemble'):
        xgb_models = train_xgb_quantile_models(
            X_train_xgb, train_df[TARGET_COL], X_val_xgb, val_df[TARGET_COL],
            quantiles, params=xgb_params,
            n_estimators_max=MAX_BOOST_ROUNDS,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            sample_weight_train=train_w,
            sample_weight_valid=val_w,
        )
        pred_map_xgb = predict_xgb_quantiles(xgb_models, X_test_xgb)
        if framework == 'XGBoost':
            pred_map = pred_map_xgb
    
    if framework in ('LightGBM', 'WeightedEnsemble'):
        lgbm_models = train_lgbm_quantile_models(
            X_train_lgb, train_df[TARGET_COL], X_val_lgb, val_df[TARGET_COL],
            CAT_COLS, quantiles, params=lgbm_params,
            n_estimators_max=MAX_BOOST_ROUNDS,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            sample_weight_train=train_w,
            sample_weight_valid=val_w,
        )
        pred_map_lgbm = predict_lgbm_quantiles(lgbm_models, X_test_lgb)
        if framework == 'LightGBM':
            pred_map = pred_map_lgbm
    
    if framework == 'WeightedEnsemble':
        pred_map = blend_prediction_maps([pred_map_xgb, pred_map_lgbm], weights=[0.55, 0.45])
    
    metrics = evaluate_point_and_interval(test_df[TARGET_COL].to_numpy(), pred_map, is_log_target=is_log_target)
    metrics_df = pd.DataFrame([{'framework': framework, **metrics}])
    metrics_df = metrics_df.sort_values(['MAPE_pct', 'MAE']).reset_index(drop=True)
    
    median_key = 'median' if 'median' in pred_map else 'q50'
    median_pred = np.asarray(pred_map[median_key], dtype=float)
    if is_log_target:
        if float(np.nanmedian(test_df[TARGET_COL].to_numpy())) < 10:
            median_pred = np.power(10.0, median_pred)
        else:
            median_pred = np.exp(median_pred)
    diag_df = per_make_model_mape_df(test_df, median_pred, min_rows=5)
    
    return metrics_df, pred_map, diag_df

## 8) Run Experiment Ladder

Execute all experiments with partial-run caching.

In [12]:
experiment_results: list[pd.DataFrame] = []
experiment_diags: dict[str, pd.DataFrame] = {}
experiment_pred_maps: dict[str, dict] = {}

for experiment in EXPERIMENTS:
    name = experiment['experiment_name']
    partial_metrics_path = PARTIAL_DIR / f"metrics__{name}.csv"
    partial_diag_path = PARTIAL_DIR / f"diag__{name}.csv"

    if partial_metrics_path.exists() and partial_diag_path.exists():
        print(f"Loading cached: {name}")
        metrics_df = pd.read_csv(partial_metrics_path)
        diag_df = pd.read_csv(partial_diag_path)
        pred_map = None
    else:
        print(f"Starting: {name}")
        metrics_df, pred_map, diag_df = run_single_experiment(experiment)
        metrics_df.to_csv(partial_metrics_path, index=False)
        diag_df.to_csv(partial_diag_path, index=False)
        print(f"Completed: {name}")

    metrics_df = metrics_df.copy()
    metrics_df.insert(0, 'experiment_name', name)
    metrics_df.insert(1, 'use_weights', experiment['use_weights'])
    metrics_df.insert(2, 'use_tuned', experiment['use_tuned'])
    metrics_df.insert(3, 'quantile_count', len(experiment['quantiles']))
    experiment_results.append(metrics_df)
    experiment_diags[name] = diag_df
    experiment_pred_maps[name] = pred_map

experiment_results_df = pd.concat(experiment_results, ignore_index=True)
display(experiment_results_df.sort_values(['MAPE_pct', 'MAE']).reset_index(drop=True))

NameError: name 'EXPERIMENTS' is not defined

## 9) Shortlist & Winner Selection

Select the best experiment and compare to the 07a baseline.

In [13]:
cache_out_dir = OUT_DIR if 'OUT_DIR' in globals() else Path('/home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments')
shortlist_path = cache_out_dir / 'v2_experiment_shortlist.csv'
full_results_path = cache_out_dir / 'v2_experiment_results_full.csv'
baseline_dir = BASELINE_DIR if 'BASELINE_DIR' in globals() else Path('/home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07a_dataset_baselines')
baseline_summary_path = baseline_dir / 'baseline_results_summary.csv'

if 'experiment_results_df' not in globals():
    if full_results_path.exists():
        experiment_results_df = pd.read_csv(full_results_path)
        print(f'Loaded cached experiment results from: {full_results_path}')
    elif shortlist_path.exists():
        experiment_results_df = pd.read_csv(shortlist_path)
        print(f'Loaded cached shortlist as experiment results from: {shortlist_path}')
    else:
        raise FileNotFoundError(
            'No cached experiment results found. Run the experiment ladder once or restore '
            f'{full_results_path.name} / {shortlist_path.name}.'
        )

if 'shortlist_df' not in globals():
    if shortlist_path.exists():
        shortlist_df = pd.read_csv(shortlist_path)
        print(f'Loaded cached shortlist from: {shortlist_path}')
    else:
        shortlist_df = (
            experiment_results_df
            .sort_values(['experiment_name', 'MAPE_pct', 'MAE'])
            .groupby('experiment_name', as_index=False)
            .first()
            .sort_values(['MAPE_pct', 'MAE'])
            .reset_index(drop=True)
        )
        print('Built shortlist from experiment_results_df in memory.')

display(shortlist_df)

best_v2 = shortlist_df.sort_values(['MAPE_pct', 'MAE']).iloc[0].to_dict()
best_exp_name = best_v2['experiment_name']
best_mape = best_v2['MAPE_pct']

if 'baseline_mape' not in globals():
    baseline_mape = 11.33
    if baseline_summary_path.exists():
        baseline_df = pd.read_csv(baseline_summary_path)
        if {'dataset_tag', 'split_name', 'target_col'}.issubset(baseline_df.columns):
            dataset_tag = globals().get('DATASET_TAG', '2026-06-03_008')
            split_name = globals().get('SPLIT_NAME', 'price_stratified')
            target_col = globals().get('TARGET_COL', 'price_egp_log')
            baseline_row = baseline_df[
                (baseline_df['dataset_tag'] == dataset_tag) &
                (baseline_df['split_name'] == split_name) &
                (baseline_df['target_col'] == target_col)
            ]
            if len(baseline_row):
                baseline_mape = float(baseline_row.iloc[0]['MAPE_pct'])

print(f"Best V2 experiment: {best_exp_name}")
print(f"Best V2 MAPE: {best_mape:.2f}%")
print(f"07a baseline MAPE: {baseline_mape:.2f}%")
delta = baseline_mape - best_mape
print(f"Delta: {delta:.2f} pp {'(improved)' if delta > 0 else '(no improvement)'}")

best_v2

Loaded cached experiment results from: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments/v2_experiment_results_full.csv
Loaded cached shortlist from: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments/v2_experiment_shortlist.csv


,experiment_name,use_weights,use_tuned,quantile_count,framework,MAE,RMSE,R2,MAPE_pct,Within_10pct,Within_15pct,Coverage_80_pct,Coverage_90_pct,Mean_width_pct
0,ensemble_baseline_5q,False,False,5,WeightedEnsemble,"151,004.7090","602,399.1851",0.8887,11.2937,61.6400,76.8800,73.5200,86.5200,36.2937
1,ensemble_tuned_5q,False,True,5,WeightedEnsemble,"153,625.6299","608,655.6234",0.8864,11.4448,61.2800,76.8000,74.0400,86.4800,35.3316
2,xgb_tuned_weighted_5q,True,True,5,XGBoost,"160,613.7486","631,653.5535",0.8776,11.6137,60.0400,75.0000,69.6800,85.4000,33.5699


Best V2 experiment: ensemble_baseline_5q
Best V2 MAPE: 11.29%
07a baseline MAPE: 11.33%
Delta: 0.04 pp (improved)


{'experiment_name': 'ensemble_baseline_5q',
 'use_weights': False,
 'use_tuned': False,
 'quantile_count': 5,
 'framework': 'WeightedEnsemble',
 'MAE': 151004.70901961936,
 'RMSE': 602399.1851153284,
 'R2': 0.8886956322366752,
 'MAPE_pct': 11.29368524251808,
 'Within_10pct': 61.63999999999999,
 'Within_15pct': 76.88000000000001,
 'Coverage_80_pct': 73.52,
 'Coverage_90_pct': 86.52,
 'Mean_width_pct': 36.29374767052723}

## 10) Cross-Validation on Best Experiment

Run 5-fold CV on the best experiment using the same framework and params.

For fair comparison with `07a` and `07b`, this CV step runs on the **full reconstructed dataset** (`df_full`), not only the train+validation subset. This keeps the per-make-model denominator aligned with the earlier notebooks when exporting cross-validation diagnostics.

In [14]:
if 'best_v2' not in globals():
    cache_out_dir = OUT_DIR if 'OUT_DIR' in globals() else Path('/home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments')
    shortlist_path = cache_out_dir / 'v2_experiment_shortlist.csv'
    if not shortlist_path.exists():
        raise FileNotFoundError(f'Cached shortlist not found at: {shortlist_path}')
    shortlist_df = pd.read_csv(shortlist_path)
    best_v2 = shortlist_df.sort_values(['MAPE_pct', 'MAE']).iloc[0].to_dict()
    best_exp_name = best_v2['experiment_name']
    print(f'Loaded cached winner from: {shortlist_path}')
else:
    best_exp_name = best_v2['experiment_name']

if 'df_full' not in globals():
    df_full = load_processed_dataset(DATASET_TAG)
    print(f'Loaded df_full from dataset tag: {DATASET_TAG}')

if 'FEATURE_COLS' not in globals() or 'CAT_COLS' not in globals():
    _, CAT_COLS, FEATURE_COLS = build_feature_spec(FEATURE_LIST)

xgb_tuned_params = globals().get('xgb_tuned_params', {})
lgbm_tuned_params = globals().get('lgbm_tuned_params', {})

best_framework = best_v2['framework']
use_tuned_for_cv = bool(best_v2['use_tuned'])

if use_tuned_for_cv and (not xgb_tuned_params or not lgbm_tuned_params) and best_framework == 'WeightedEnsemble':
    raise RuntimeError('The cached winner requires tuned params, but tuned params are not loaded in memory. Run the tuning cells once or switch to an untuned cached winner.')
if use_tuned_for_cv and best_framework == 'XGBoost' and not xgb_tuned_params:
    raise RuntimeError('The cached winner requires XGBoost tuned params, but they are not loaded in memory. Run the tuning cells once.')
if use_tuned_for_cv and best_framework == 'LightGBM' and not lgbm_tuned_params:
    raise RuntimeError('The cached winner requires LightGBM tuned params, but they are not loaded in memory. Run the tuning cells once.')

if best_framework == 'WeightedEnsemble':
    cv_framework = 'WeightedEnsemble'
    cv_params = {
        'xgb': ({**XGB_BASE_PARAMS, **xgb_tuned_params, 'seed': RANDOM_STATE} if use_tuned_for_cv else {**XGB_BASE_PARAMS, 'seed': RANDOM_STATE}),
        'lgbm': ({**LGBM_BASE_PARAMS, **lgbm_tuned_params, 'seed': RANDOM_STATE} if use_tuned_for_cv else {**LGBM_BASE_PARAMS, 'seed': RANDOM_STATE}),
        'weights': [0.55, 0.45],
    }
elif best_framework == 'LightGBM':
    cv_framework = 'LightGBM'
    cv_params = ({**LGBM_BASE_PARAMS, **lgbm_tuned_params, 'seed': RANDOM_STATE} if use_tuned_for_cv else {**LGBM_BASE_PARAMS, 'seed': RANDOM_STATE})
else:
    cv_framework = 'XGBoost'
    cv_params = ({**XGB_BASE_PARAMS, **xgb_tuned_params, 'seed': RANDOM_STATE} if use_tuned_for_cv else {**XGB_BASE_PARAMS, 'seed': RANDOM_STATE})

print(f'Running 5-fold CV for best experiment: {best_exp_name}')
print(f'CV framework: {cv_framework}')
print(f'CV dataset rows: {len(df_full):,} (full dataset for fair comparison with 07a/07b)')

cv_per_mm_df, cv_overall_metrics = run_kfold_per_make_model_cv(
    df_full,
    feature_cols=FEATURE_COLS,
    cat_cols=CAT_COLS,
    target_col=TARGET_COL,
    framework=cv_framework,
    params=cv_params,
    n_splits=5,
    random_state=RANDOM_STATE,
    min_rows=5,
)

print('\nCV overall metrics:')
for k, v in cv_overall_metrics.items():
    print(f'  {k}: {v}')

print('\nBest 20 CV per-make-model (lowest MAPE):')
display(cv_per_mm_df.sort_values('MAPE_pct', ascending=True).head(20))

print('Worst 20 CV per-make-model (highest MAPE):')
display(cv_per_mm_df.sort_values('MAPE_pct', ascending=False).tail(20))

Running 5-fold CV for best experiment: ensemble_baseline_5q
CV framework: WeightedEnsemble
CV dataset rows: 12,498 (full dataset for fair comparison with 07a/07b)

CV overall metrics:
  MAE: 154040.7402990791
  RMSE: 505786.3749423331
  R2: 0.9223273206165608
  MAPE_pct: 11.832901907986267
  Within_10pct: 61.769883181309005
  Within_15pct: 77.00432069131061

Best 20 CV per-make-model (lowest MAPE):


,make,model,n,MAPE_pct,MAE,R2,mean_price
0,Jetour,X90,11,2.7854,"41,319.3439",0.7084,"1,501,363.6364"
1,Cupra,Formentor,23,3.1805,"63,388.2861",0.7705,"2,133,435.1739"
2,Geely,GX3 Pro,8,3.2177,"25,519.2094",0.7843,"818,750.0000"
3,BAIC,BJ30,12,3.3324,"56,339.4636",0.2094,"1,694,575.0000"
4,Kia,Xceed,23,3.3966,"43,394.0395",0.8390,"1,316,086.9565"
5,Jetour,X95,6,3.5932,"44,916.8245",0.5795,"1,233,333.3333"
6,Jetour,X70 Plus,9,3.7852,"48,846.8057",-0.1377,"1,297,430.5556"
7,MG,7,10,4.1647,"69,476.9082",0.5245,"1,638,500.0000"
8,Chery,Arrizo 6,11,4.2804,"43,763.2781",0.0274,"1,029,274.5455"
9,Soueast,S05,9,4.2861,"44,812.4937",-0.6054,"1,071,111.1111"


Worst 20 CV per-make-model (highest MAPE):


,make,model,n,MAPE_pct,MAE,R2,mean_price
19,Rox,01,17,5.2718,"174,805.6809",-0.2744,"3,723,823.5294"
18,Chery,Tiggo 3,59,5.1583,"31,768.6719",0.8670,"656,864.4068"
17,Hyundai,Accent RB,44,4.9343,"33,674.9642",0.8849,"701,022.7273"
16,Citroën,C5 Aircross,17,4.7834,"55,498.4238",0.9239,"1,287,061.7647"
15,Jetour,Dashing,13,4.7777,"67,931.1351",-0.9532,"1,394,461.5385"
14,Jetour,T2,32,4.6063,"77,179.9365",0.3320,"1,923,906.2500"
13,Cupra,Leon,27,4.5969,"84,550.7898",-0.2524,"1,878,111.1111"
12,MG,ZS,62,4.5555,"48,183.3874",0.8788,"1,036,387.0968"
11,Chery,Tiggo 7,51,4.5505,"40,502.8600",0.7463,"911,176.4706"
10,Suzuki,Baleno,8,4.4087,"37,167.1697",0.8358,"846,875.0000"


## 11) Best & Worst per-Make-Model by MAPE

Inspect the best experiment at the make-model diagnostic level.

In [15]:
cache_out_dir = OUT_DIR if 'OUT_DIR' in globals() else Path('/home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments')

if 'best_v2' not in globals():
    shortlist_df = pd.read_csv(cache_out_dir / 'v2_experiment_shortlist.csv')
    best_v2 = shortlist_df.sort_values(['MAPE_pct', 'MAE']).iloc[0].to_dict()
    best_exp_name = best_v2['experiment_name']

best_diag_path = cache_out_dir / f'best_make_model_mape__{best_exp_name}.csv'

if 'experiment_diags' in globals() and best_exp_name in experiment_diags:
    best_diag = experiment_diags[best_exp_name].copy()
elif best_diag_path.exists():
    best_diag = pd.read_csv(best_diag_path)
    print(f'Loaded cached holdout diagnostics from: {best_diag_path}')
else:
    raise FileNotFoundError(f'No cached holdout diagnostics found for {best_exp_name} at: {best_diag_path}')

print('Best 20 per-make-model (lowest MAPE):')
display(best_diag.sort_values('MAPE_pct', ascending=True).head(20))

print('Worst 20 per-make-model (highest MAPE):')
display(best_diag.sort_values('MAPE_pct', ascending=False).head(20))

Loaded cached holdout diagnostics from: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments/best_make_model_mape__ensemble_baseline_5q.csv
Best 20 per-make-model (lowest MAPE):


,make,model,n_test,MAPE_pct
159,Land Rover,Range Rover Velar,5,3.2440
158,Chery,Tiggo 3,11,3.3296
157,MG,ZS,12,3.6767
156,Audi,A3,8,3.7545
155,Skoda,Octavia A7,5,3.8976
154,MG,6,12,4.0234
153,Mercedes,GLC300,14,4.2108
152,Skoda,Kodiaq,24,4.2350
151,Hyundai,Tucson,25,4.3236
150,Mercedes,CLA180,5,4.6039


Worst 20 per-make-model (highest MAPE):


,make,model,n_test,MAPE_pct
0,Mazda,323,6,43.7698
1,Chevrolet,Pickup,5,42.5834
2,Suzuki,Swift,8,37.2115
3,Peugeot,504,6,36.2034
4,Peugeot,405,9,31.2386
5,Mercedes,200,8,26.4276
6,Honda,Civic,10,26.2382
7,Fiat,128,11,23.7940
8,Mercedes,G63,6,23.3908
9,Mercedes,S500,5,22.8372


## 12) Best & Worst per-Make-Model by R²

Same winning experiment, sorted by per-make-model R² instead of MAPE.

In [16]:
cache_out_dir = OUT_DIR if 'OUT_DIR' in globals() else Path('/home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments')

if 'best_v2' not in globals():
    shortlist_df = pd.read_csv(cache_out_dir / 'v2_experiment_shortlist.csv')
    best_v2 = shortlist_df.sort_values(['MAPE_pct', 'MAE']).iloc[0].to_dict()
    best_exp_name = best_v2['experiment_name']

r2_diag_path = cache_out_dir / f'best_make_model_r2__{best_exp_name}.csv'

if r2_diag_path.exists():
    r2_diag_df = pd.read_csv(r2_diag_path)
    print(f'Loaded cached R² diagnostics from: {r2_diag_path}')
else:
    best_pred_map = experiment_pred_maps.get(best_exp_name) if 'experiment_pred_maps' in globals() else None
    if not best_pred_map:
        raise FileNotFoundError(
            f'Cached R² diagnostics not found at: {r2_diag_path}. '
            'Run the holdout prediction cell/experiment ladder once before rerunning this cell.'
        )

    test_df_for_r2 = test_df.reset_index(drop=True)
    median_key = 'median' if 'median' in best_pred_map else 'q50'
    median_pred = np.asarray(best_pred_map.get(median_key, np.array([])), dtype=float)

    if is_log_target and len(median_pred) > 0:
        if float(np.nanmedian(test_df_for_r2[TARGET_COL].to_numpy())) < 10:
            median_pred = np.power(10.0, median_pred)
        else:
            median_pred = np.exp(median_pred)

    r2_rows = []
    for (make, model), group in test_df_for_r2.groupby(['make', 'model'], dropna=False):
        if len(group) < 5:
            continue
        positions = group.index.to_numpy()
        y_true = group['price_egp'].to_numpy()
        y_pred = median_pred[positions]
        r2_rows.append({
            'make': make,
            'model': model,
            'n_test': len(group),
            'R2': r2_score(y_true, y_pred),
            'MAPE_pct': float(np.mean(np.abs(y_true - y_pred) / y_true) * 100),
        })

    r2_diag_df = pd.DataFrame(r2_rows).sort_values('R2', ascending=False).reset_index(drop=True)

print('Best 20 per-make-model by R²:')
display(r2_diag_df.sort_values('R2', ascending=False).head(20))
print('Worst 20 per-make-model by R²:')
display(r2_diag_df.sort_values('R2', ascending=True).head(20))

Loaded cached R² diagnostics from: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments/best_make_model_r2__ensemble_baseline_5q.csv
Best 20 per-make-model by R²:


,make,model,n_test,R2,MAPE_pct
0,Mercedes,E200,35,0.9838,12.4384
1,Peugeot,408,5,0.9758,8.2167
2,Volkswagen,Golf,12,0.9751,10.6488
3,Audi,A3,8,0.9730,3.7545
4,Opel,Corsa,15,0.9726,14.6913
5,Land Rover,Range Rover Evoque,8,0.9717,4.9486
6,BMW,318i,8,0.9679,8.5086
7,Skoda,Octavia,18,0.9649,6.3698
8,Hyundai,Accent,26,0.9589,9.4067
9,Land Rover,Range Rover Velar,5,0.9588,3.2440


Worst 20 per-make-model by R²:


,make,model,n_test,R2,MAPE_pct
159,Peugeot,405,9,-9.5167,31.2386
158,Kia,Spectra,5,-7.4049,18.5587
157,Mercedes,GLE450,7,-1.8147,14.2134
156,Toyota,Fortuner,9,-0.7878,12.5412
155,Geely,Emgrand 7,9,-0.7592,11.9238
154,Fiat,128,11,-0.6676,23.7940
153,Renault,Fluence,7,-0.6131,12.4121
152,Fiat,Regata,5,-0.5159,22.2518
151,Opel,Grandland,8,-0.2723,15.9367
150,Daewoo,Lanos 2,5,-0.1202,15.6437


## 13) Per-Tier Evaluation (Full-Dataset CV)

Compute metrics by price tier for the selected 07c winner using **out-of-fold predictions from the same 5-fold cross-validation setup used in Section 10**.

This section is **cache-friendly**:
- it first loads cached per-tier CSV and OOF predictions if present,
- otherwise it reuses the Section 10 CV configuration and reruns that same CV once to capture OOF predictions,
- it does **not** rerun Optuna or the holdout experiment ladder.

In [18]:
import importlib
import v2_experiment_utils as v2_utils

v2_utils = importlib.reload(v2_utils)

cache_out_dir = OUT_DIR if 'OUT_DIR' in globals() else Path('/home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments')

if 'best_v2' not in globals():
    shortlist_df = pd.read_csv(cache_out_dir / 'v2_experiment_shortlist.csv')
    best_v2 = shortlist_df.sort_values(['MAPE_pct', 'MAE']).iloc[0].to_dict()
    best_exp_name = best_v2['experiment_name']
else:
    best_exp_name = best_v2['experiment_name']

per_tier_cv_path = cache_out_dir / f'per_tier_cv_metrics__{best_exp_name}.csv'
oof_preds_path = cache_out_dir / f'oof_median_preds__{best_exp_name}.csv'
FORCE_RECOMPUTE_PER_TIER = False

TIER_ORDER = [
    'Economy (< 500K)',
    'Standard (500K-1.5M)',
    'Luxury (1.5M-4M)',
    'Ultra-luxury (> 4M)',
]

def assign_price_tier(price_egp: float) -> str:
    if price_egp < 500_000:
        return 'Economy (< 500K)'
    if price_egp < 1_500_000:
        return 'Standard (500K-1.5M)'
    if price_egp < 4_000_000:
        return 'Luxury (1.5M-4M)'
    return 'Ultra-luxury (> 4M)'

def compute_per_tier_metrics(eval_df: pd.DataFrame, median_pred_egp: np.ndarray) -> pd.DataFrame:
    eval_df = eval_df.reset_index(drop=True).copy()
    eval_df['price_tier'] = eval_df['price_egp'].apply(assign_price_tier)
    y_true_all = eval_df['price_egp'].to_numpy(dtype=float)
    ape_all = np.abs(y_true_all - median_pred_egp) / np.maximum(np.abs(y_true_all), 1e-9)

    rows = []
    for tier in TIER_ORDER:
        mask = eval_df['price_tier'].eq(tier).to_numpy()
        if mask.sum() == 0:
            continue
        y_true = y_true_all[mask]
        y_pred = median_pred_egp[mask]
        ape = ape_all[mask]
        rows.append({
            'tier': tier,
            'n': int(mask.sum()),
            'MAPE_pct': float(np.mean(ape) * 100),
            'MAE': float(np.mean(np.abs(y_true - y_pred))),
            'R2': float(r2_score(y_true, y_pred)) if len(y_true) >= 2 else np.nan,
            'Within_10pct': float(np.mean(ape <= 0.10) * 100),
            'Within_15pct': float(np.mean(ape <= 0.15) * 100),
        })

    out = pd.DataFrame(rows)
    if not out.empty:
        out['tier'] = pd.Categorical(out['tier'], categories=TIER_ORDER, ordered=True)
        out = out.sort_values('tier').reset_index(drop=True)
        out['tier'] = out['tier'].astype(str)
        out['MAPE_pct'] = out['MAPE_pct'].round(4)
        out['MAE'] = out['MAE'].round(2)
        out['R2'] = out['R2'].round(4)
        out['Within_10pct'] = out['Within_10pct'].round(2)
        out['Within_15pct'] = out['Within_15pct'].round(2)
    return out

if per_tier_cv_path.exists() and oof_preds_path.exists() and not FORCE_RECOMPUTE_PER_TIER:
    per_tier_cv_df = pd.read_csv(per_tier_cv_path)
    oof_preds = pd.read_csv(oof_preds_path)['oof_median_pred'].to_numpy()
    print(f'Loaded cached CV per-tier metrics from: {per_tier_cv_path}')
else:
    if 'df_full' not in globals():
        df_full = load_processed_dataset(DATASET_TAG)
        print(f'Loaded df_full from dataset tag: {DATASET_TAG}')

    if 'FEATURE_COLS' not in globals() or 'CAT_COLS' not in globals():
        _, CAT_COLS, FEATURE_COLS = build_feature_spec(FEATURE_LIST)

    if 'cv_framework' not in globals() or 'cv_params' not in globals():
        xgb_tuned_params = globals().get('xgb_tuned_params', {})
        lgbm_tuned_params = globals().get('lgbm_tuned_params', {})
        best_framework = best_v2['framework']
        use_tuned_for_cv = bool(best_v2['use_tuned'])

        if use_tuned_for_cv and (not xgb_tuned_params or not lgbm_tuned_params) and best_framework == 'WeightedEnsemble':
            raise RuntimeError('The cached winner requires tuned params, but tuned params are not loaded in memory. Run the tuning cells once.')
        if use_tuned_for_cv and best_framework == 'XGBoost' and not xgb_tuned_params:
            raise RuntimeError('The cached winner requires XGBoost tuned params, but they are not loaded in memory. Run the tuning cells once.')
        if use_tuned_for_cv and best_framework == 'LightGBM' and not lgbm_tuned_params:
            raise RuntimeError('The cached winner requires LightGBM tuned params, but they are not loaded in memory. Run the tuning cells once.')

        if best_framework == 'WeightedEnsemble':
            cv_framework = 'WeightedEnsemble'
            cv_params = {
                'xgb': ({**XGB_BASE_PARAMS, **xgb_tuned_params, 'seed': RANDOM_STATE} if use_tuned_for_cv else {**XGB_BASE_PARAMS, 'seed': RANDOM_STATE}),
                'lgbm': ({**LGBM_BASE_PARAMS, **lgbm_tuned_params, 'seed': RANDOM_STATE} if use_tuned_for_cv else {**LGBM_BASE_PARAMS, 'seed': RANDOM_STATE}),
                'weights': [0.55, 0.45],
            }
        elif best_framework == 'LightGBM':
            cv_framework = 'LightGBM'
            cv_params = ({**LGBM_BASE_PARAMS, **lgbm_tuned_params, 'seed': RANDOM_STATE} if use_tuned_for_cv else {**LGBM_BASE_PARAMS, 'seed': RANDOM_STATE})
        else:
            cv_framework = 'XGBoost'
            cv_params = ({**XGB_BASE_PARAMS, **xgb_tuned_params, 'seed': RANDOM_STATE} if use_tuned_for_cv else {**XGB_BASE_PARAMS, 'seed': RANDOM_STATE})

    print('Running the Section 10 CV recipe once to capture OOF predictions for per-tier metrics...')

    cv_result = v2_utils.run_kfold_per_make_model_cv(
        df_full,
        feature_cols=FEATURE_COLS,
        cat_cols=CAT_COLS,
        target_col=TARGET_COL,
        framework=cv_framework,
        params=cv_params,
        n_splits=5,
        random_state=RANDOM_STATE,
        min_rows=5,
        return_oof=True,
    )

    cv_per_mm_df, cv_overall_metrics, oof_preds = cv_result
    per_tier_cv_df = compute_per_tier_metrics(df_full, oof_preds)
    per_tier_cv_df.to_csv(per_tier_cv_path, index=False)
    pd.DataFrame({'oof_median_pred': oof_preds}).to_csv(oof_preds_path, index=False)
    print(f'Saved CV per-tier metrics to: {per_tier_cv_path}')
    print(f'Saved OOF predictions to: {oof_preds_path}')

display(per_tier_cv_df)

Running the Section 10 CV recipe once to capture OOF predictions for per-tier metrics...
Saved CV per-tier metrics to: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments/per_tier_cv_metrics__ensemble_baseline_5q.csv
Saved OOF predictions to: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments/oof_median_preds__ensemble_baseline_5q.csv


,tier,n,MAPE_pct,MAE,R2,Within_10pct,Within_15pct
0,Economy (< 500K),4438,15.7613,"37,883.8700",0.6386,50.7200,66.5600
1,Standard (500K-1.5M),4877,8.9359,"77,296.9000",0.7262,70.2700,85.0500
2,Luxury (1.5M-4M),2319,9.9896,"238,435.2000",0.6631,66.8400,81.8000
3,Ultra-luxury (> 4M),864,12.9542,"957,366.1400",0.6280,56.9400,72.3400


## 14) Export Artifacts

Save experiment results, shortlist, CV outputs, diagnostics, and selected V2 recipe.

In [ ]:
cache_out_dir = OUT_DIR if 'OUT_DIR' in globals() else Path('/home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments')

if 'best_v2' not in globals():
    shortlist_df = pd.read_csv(cache_out_dir / 'v2_experiment_shortlist.csv')
    best_v2 = shortlist_df.sort_values(['MAPE_pct', 'MAE']).iloc[0].to_dict()
    best_exp_name = best_v2['experiment_name']

if 'experiment_results_df' not in globals() and (cache_out_dir / 'v2_experiment_results_full.csv').exists():
    experiment_results_df = pd.read_csv(cache_out_dir / 'v2_experiment_results_full.csv')
if 'shortlist_df' not in globals() and (cache_out_dir / 'v2_experiment_shortlist.csv').exists():
    shortlist_df = pd.read_csv(cache_out_dir / 'v2_experiment_shortlist.csv')
if 'best_diag' not in globals() and (cache_out_dir / f'best_make_model_mape__{best_exp_name}.csv').exists():
    best_diag = pd.read_csv(cache_out_dir / f'best_make_model_mape__{best_exp_name}.csv')
if 'r2_diag_df' not in globals() and (cache_out_dir / f'best_make_model_r2__{best_exp_name}.csv').exists():
    r2_diag_df = pd.read_csv(cache_out_dir / f'best_make_model_r2__{best_exp_name}.csv')

if 'experiment_results_df' in globals():
    experiment_results_df.to_csv(cache_out_dir / 'v2_experiment_results_full.csv', index=False)
if 'shortlist_df' in globals():
    shortlist_df.to_csv(cache_out_dir / 'v2_experiment_shortlist.csv', index=False)
if 'best_diag' in globals():
    best_diag.to_csv(cache_out_dir / f'best_make_model_mape__{best_exp_name}.csv', index=False)
if 'r2_diag_df' in globals():
    r2_diag_df.to_csv(cache_out_dir / f'best_make_model_r2__{best_exp_name}.csv', index=False)

# CV artifacts
if 'cv_per_mm_df' in globals():
    cv_per_mm_df.to_csv(cache_out_dir / 'cv_per_make_model_diagnostics.csv', index=False)
if 'cv_overall_metrics' in globals():
    cv_overall_df = pd.DataFrame([cv_overall_metrics])
    cv_overall_df.to_csv(cache_out_dir / 'cv_overall_metrics.csv', index=False)

# Per-tier CV artifacts
if 'per_tier_cv_df' in globals() and 'per_tier_cv_path' in globals():
    per_tier_cv_df.to_csv(per_tier_cv_path, index=False)
    print('Saved CV per-tier metrics to:', per_tier_cv_path)
if 'oof_preds' in globals() and 'oof_preds_path' in globals():
    pd.DataFrame({'oof_median_pred': oof_preds}).to_csv(oof_preds_path, index=False)
    print('Saved OOF predictions to:', oof_preds_path)

# Selected V2 recipe
feature_cols_for_export = FEATURE_COLS if 'FEATURE_COLS' in globals() else build_feature_spec(FEATURE_LIST)[2]
recipe_df = pd.DataFrame([
    {
        'dataset_tag': DATASET_TAG,
        'target_col': TARGET_COL,
        'split_name': SPLIT_NAME,
        'feature_list': ','.join(feature_cols_for_export),
        'best_experiment_name': best_exp_name,
        'best_framework': best_v2['framework'],
        'best_quantile_count': best_v2['quantile_count'],
        'uses_sample_weights': best_v2['use_weights'],
        'uses_tuned_params': best_v2['use_tuned'],
    }
])
recipe_df.to_csv(cache_out_dir / 'selected_v2_recipe.csv', index=False)
print('Saved artifacts to:', cache_out_dir)

Saved artifacts to: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments
